# 172 — Razonamiento y cómputo en tiempo de inferencia

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Test-time compute**: gastar más cómputo al responder, no solo al entrenar.
Cuatro escalones:

1. **CoT** (arXiv:2201.11903): generar pasos intermedios; cada token es cómputo
   condicionado extra. Emergente con la escala; la cadena NO es garantía de
   explicación fiel.
2. **Self-consistency** (arXiv:2203.11171): muestrear k cadenas con temperatura,
   extraer respuestas, votar la mayoría. Marginaliza sobre caminos de
   razonamiento.
3. **Buscar + verificar**: best-of-N con verificador (ORM/PRM),
   Tree-of-Thoughts con retroceso — búsqueda clásica sobre estados generados.
4. **Modelos razonadores (o1/R1)**: RL sobre trazas largas premiando respuestas
   verificables; precisión ~ log(cómputo de pensamiento).

Hallazgo de Snell et al. (arXiv:2408.03314): asignar cómputo *adaptativo a la
dificultad* puede rendir más que un modelo ~14× mayor — pero en preguntas
fáciles no aporta y en las imposibles por conocimiento tampoco.


## 🧮 Ejemplo clave (self-consistency)

Si cada muestra acierta con p = 0.6 y los errores se reparten:
`P(mayoría de 5 correcta) = 10·0.6³·0.4² + 5·0.6⁴·0.4 + 0.6⁵ ≈ 0.683`.
Votar sube 60 % → 68.3 % (k=11 → ~75 %), con coste lineal en k.

La trampa: con error sistemático p = 0.4 hacia una misma respuesta, la mayoría
de 5 baja a ~31.7 %. **Votar amplifica lo que domine: la señal o el sesgo.**


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("frontier", seed=172)
show(result)


## Reflexión

1. ¿En qué condición exacta self-consistency empeora la precisión respecto a
   una sola muestra, y cómo la detectarías empíricamente?
2. Best-of-N con verificador optimiza contra el juez: ¿qué experimento
   distinguiría "el juez selecciona soluciones correctas" de "las soluciones
   explotan sesgos del juez"?
3. "Nuestro modelo razona: su precisión sube con más tokens de pensamiento".
   ¿Qué madurez (`current` / `needs_replication` / `unverified`) darías a esa
   claim si el paper solo reporta benchmarks de matemática con verificador?
